# Phase 5 — анализ результатов FLV-эксперимента

Воспроизводимый ноутбук поверх `results.xlsx`, собранного `exp-analyze`. Применяется для повторных расчётов в редактуре главы 4 ПЗ.

**Контекст.** ВКР Катальшова Д.А. (К2-81Б, 12.03.01 «Информационно-измерительная техника и технологии», МФ МГТУ Бауман). Парное сравнение наивного baseline'а и метода функционально-логической верификации (FLV) на 8 сценариях термокамеры PT100 (1 корректный + 7 инъекций), батч 50 прогонов на сценарий, итого ~400 прогонов.

**Что делаем:**
1. Читаем `results.xlsx`.
2. Воспроизводим сводку метрик детекции.
3. Прогоняем McNemar-тест и непрерывные тесты с CI.
4. Строим публикационные графики.

**Зависимости:** `pip install -e ".[notebook,dev]"` из `05_Эксперименты/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

RESULTS = Path('../results.xlsx')
RUNS = Path('../runs')
FIGURES = Path('../figures')

assert RESULTS.exists(), f'Не найден {RESULTS}. Запусти exp-run и exp-analyze сначала.'

## 1. Загрузка данных

In [ ]:
raw = pd.read_excel(RESULTS, sheet_name='raw')
summary = pd.read_excel(RESULTS, sheet_name='summary')
by_scenario = pd.read_excel(RESULTS, sheet_name='by_scenario')
paired = pd.read_excel(RESULTS, sheet_name='paired_outcomes', index_col=0)
meta = pd.read_excel(RESULTS, sheet_name='meta')

print(f'Прогонов: {len(raw)} · Сценариев: {raw["scenario_id"].nunique()}')
summary

## 2. Парная 2×2 таблица McNemar

In [ ]:
from experiments.stats import mcnemar_test

paired

In [ ]:
res = mcnemar_test(paired)
display(Markdown(
    f'**McNemar test:** statistic = {res.statistic:.4g}, '
    f'p-value = {res.pvalue:.4g}, '
    f'{"**значимо**" if res.significant else "не значимо"} '
    f'на α=0.05 ({res.meta})'
))

## 3. Производительность — overhead

In [ ]:
from experiments.stats import bootstrap_ci, cohen_d, paired_continuous_test

overhead = raw['overhead_ratio'].dropna().values
norm, loc = paired_continuous_test(overhead - 1.0, 'overhead − 1')
d = cohen_d(overhead - 1.0)
_, lo, hi = bootstrap_ci(overhead, statistic=np.mean)

display(Markdown(
    f"- mean overhead = **{overhead.mean():.3f}**, std = {overhead.std(ddof=1):.3f}\n"
    f"- 95% CI mean: [{lo:.3f}; {hi:.3f}]\n"
    f"- Cohen's d = {d:.3f}\n"
    f"- {loc.name}: p = {loc.pvalue:.4g} "
    f"({'значимо' if loc.significant else 'не значимо'})"
))

## 4. По сценариям

In [ ]:
by_scenario.style.format({
    'baseline_TPR': '{:.3f}', 'baseline_FPR': '{:.3f}', 'baseline_F1': '{:.3f}',
    'flv_TPR': '{:.3f}', 'flv_FPR': '{:.3f}', 'flv_F1': '{:.3f}',
    'overhead_ratio_mean': '{:.3f}',
})

## 5. Публикационные графики

Строим прямо здесь поверх `raw` / `summary` для интерактивной редактуры.

In [ ]:
%matplotlib inline
from experiments.plots import plot_metrics_bar, plot_confusion_paired

FIGURES.mkdir(exist_ok=True)
plot_metrics_bar(summary, FIGURES)
plot_confusion_paired(summary, FIGURES)
print(f'Фигуры сохранены в {FIGURES}/')

## 6. Чек-лист готовности к написанию главы 4 ПЗ

- [ ] `results.xlsx` пересобран (`exp-analyze`)
- [ ] `stat_report.md` сгенерирован (`exp-stats`)
- [ ] Все 7 фигур в `figures/` есть и подписаны
- [ ] McNemar p-value зафиксировано
- [ ] Все 95% CI отображены в таблицах
- [ ] Δbias по корректным/инъекционным прогонам сосчитан